In [1]:
import lxml.html as l
import requests
from random import random
from random import choice
from time import sleep
import pandas as pd
from os import listdir
from os import makedirs
from concurrent.futures import ThreadPoolExecutor

In [2]:
home = 'https://www.cifrus.ru'
url = 'https://www.cifrus.ru/catalog/smartfony'
param = '/page-'

f = open('cookie.txt', 'r', encoding='utf8')
headers = {
    'Cookie': f.read(),
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/118.0'
}
f.close()

f = open('proxies.txt', 'r', encoding='utf8')
proxies = {
    'http': choice(f.readlines())
}
f.close()

path_to_product_url = ".//*[@class='product-layout product-pricelist col-xs-12']/div/div/div/div[@class='name']/a/@href"

product_lists_dir = 'html/product_list'
product_dir = 'html/product'

timeout = 1

In [3]:
def clone(element):
    return l.etree.fromstring(l.etree.tostring(element))


def get_product_list_name(number):
    return product_lists_dir + '/' + str(number) + '.html'


def get_product_name(number):
    return product_dir + '/' + str(number) + '.html'


def get_product_list_url(page_num):
    return url if page_num == 1 else url + param + str(page_num)


def get_product_url(href):
    return home + href + '/harakteristiki#tab-specification'


def spl(s, ch1, ch2):
    if len(s.split(ch1)) == 1:
        return s.split(ch2)
    else:
        return s.split(ch1)


def get_root(name):
    html = open(name, 'r', encoding="utf8")
    root = l.fromstring(html.read())
    html.close()
    return root

In [4]:
def crawl(crawl_url, crawl_name):
    response = requests.get(crawl_url, headers=headers, proxies=proxies)

    if response.status_code == 200:
        html = open(crawl_name, "w+", encoding='utf8')
        html.write(str(response.text))
        html.close()
        sleep(timeout + 3 * round(random(), 2))

    return crawl_url, response.status_code


crawl_product_list = lambda num: crawl(get_product_list_url(num), get_product_list_name(num))

crawl_product = lambda dct, num: crawl(dct[num], get_product_name(num))


def get_urls_from_list(list_name):
    return list(map(get_product_url, get_root(list_name).xpath(path_to_product_url)))


def get_all_urls():
    urls = []

    for product_list_name in listdir('./' + product_lists_dir):
        urls.extend(get_urls_from_list(product_lists_dir + '/' + product_list_name))

    return dict(zip(list(range(1, len(urls) + 1)), urls))

In [5]:
def make_csv(res):
    df = pd.DataFrame.from_records(res, columns=['Name', 'Val, rub', 'Product code', 'Availability', 'Reviews', 'OS', 'Width, mm', 'Height, mm', 'Thickness, mm'])
    df.to_csv('parser_res.csv')


def cast_to_float(s, par):
    try:
        return float(s)
    except ValueError:
        print('Invalid data of ' + par + ': ' + s)
        return None


def attrs_uniq(root):
    return \
        root.xpath(".//h1[@class='title-page']")[0].text[16:], \
            int(root.xpath(".//div[@class='price']")[0][0].text[:-5]), \
            int(root.xpath(".//div[@class='product-info']/ul/li/span")[0].text), \
            root.xpath(".//div[@class='product-info']/ul/li/span")[1].text, \
            int(root.xpath(".//div[@class='box-review col-lg-6 col-md-12']/a")[0].text[9:])


def attrs_from_list(root):
    os, width, height, thickness = (None, None, None, None)

    for attr in root.xpath(".//div[@class='panel-attribute']/div[@class='attr_row']"):
        if attr.xpath(".//div[@class='attr-td']/span")[0].text == 'Версия ОС на начало продаж:':
            os = attr.xpath(".//div[@class='attr_col2']/span")[0].text
        if attr.xpath(".//div[@class='attr-td']/span")[0].text == 'Размеры (ШxВxТ):':
            tmp = spl(attr.xpath(".//div[@class='attr_col2']/span")[0].text[:-3].replace(' ', '').replace(',', '.'),
                      'x', 'х')
            width = cast_to_float(tmp[0], 'width')
            height = cast_to_float(tmp[1], 'height')
            thickness = cast_to_float(tmp[2], 'thickness')

    return os, width, height, thickness


def parse(product_name):
    root = get_root(product_name)

    name, val, code, stock, review = attrs_uniq(root)
    os, width, height, thickness = attrs_from_list(root)

    return name, val, code, stock, review, os, width, height, thickness


def parser():
    res = []

    for product_name in listdir('./' + product_dir):
        res.append(parse(product_dir + '/' + product_name))

        make_csv(res)

    return res


def crawl_and_parse():
    makedirs(product_lists_dir, exist_ok=True)
    makedirs(product_dir, exist_ok=True)

    with ThreadPoolExecutor(16) as executor:
        executor.map(crawl_product_list, list(range(1, 100)))

    d = get_all_urls()

    with ThreadPoolExecutor(16) as executor:
        executor.map(lambda x: crawl_product(d, x), list(range(1, len(d) + 1)))

    parser()

In [6]:
crawl_and_parse()